# Dense Scaling Laws for Fixed-Width Addition

This notebook analyzes a **real 16-run Colab T4 pilot** built on the exact 10M-parameter JAX addition transformer.

The pilot varies:

- model size \(N\): 162,176; 963,840; 5,123,584; 10,000,000 parameters;
- training budget: 50, 125, 250, and 750 optimizer steps;
- data processed \(D\): full sequence tokens seen during training.

The primary fit target is validation answer-token cross-entropy. Greedy exact match is also reported, but it saturates at 100% and therefore cannot support a smooth power-law fit by itself.

> This notebook presents a pilot, not a final universal scaling law. The data use one random seed and the arithmetic task has a sharp transition into perfect accuracy.

## Recorded pilot result

| Steps | 162K EM | 964K EM | 5.12M EM | 10.00M EM |
|---:|---:|---:|---:|---:|
| 50 | 0.045% | 0.170% | 0.115% | 0.110% |
| 125 | 0.125% | 49.240% | 99.825% | 99.940% |
| 250 | 21.690% | 100.000% | 100.000% | 100.000% |
| 750 | 100.000% | 100.000% | 100.000% | 100.000% |

The transition moves to smaller data budgets as model size increases. Validation loss remains informative after exact match has saturated.

## 1. Set up the repository

When this notebook is opened in Colab, the following cell clones the experimental branch. After the dense-scaling work is merged, set `BRANCH = "main"`.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY = "https://github.com/marcoharuni/jax-addition-transformer.git"
BRANCH = "dense-scaling"
REPO_DIR = Path("/content/jax-addition-transformer")

if "COLAB_RELEASE_TAG" in os.environ:
    if not REPO_DIR.exists():
        subprocess.run(
            ["git", "clone", "--branch", BRANCH, "--single-branch", REPOSITORY, str(REPO_DIR)],
            check=True,
        )
    os.chdir(REPO_DIR)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", ".", "--no-deps"],
        check=True,
    )
else:
    candidate = Path.cwd()
    if not (candidate / "pyproject.toml").exists():
        raise RuntimeError("Run this notebook from the repository root.")

print("Repository:", Path.cwd())

## 2. Verify the recorded artifacts

In [ ]:
import json

results_path = Path("artifacts/dense-scaling-pilot-t4/pilot_results.json")
records = json.loads(results_path.read_text())

complete = [record for record in records if record["status"] == "complete"]
gpu_runs = [record for record in complete if record.get("backend") == "gpu"]

print("Rows:", len(records))
print("Complete:", len(complete))
print("GPU:", len(gpu_runs))

assert len(records) == 16
assert len(complete) == 16
assert len(gpu_runs) == 16

## 3. Generate the auditable table and exploratory fit

The analysis script reads only committed run artifacts. It writes:

- `pilot_table.csv`;
- `pilot_fit.json`;
- an artifact README containing the exact-match and loss matrices.

In [ ]:
subprocess.run(
    [sys.executable, "experiments/dense_scaling/analyze_pilot.py"],
    check=True,
)

fit = json.loads(
    Path("artifacts/dense-scaling-pilot-t4/pilot_fit.json").read_text()
)
print(json.dumps(fit, indent=2))

## 4. Chinchilla-style model and identifiability test

The target surface is

\[
L(N,D)
=
E
+
A\left(\frac{N}{10^6}\right)^{-\alpha}
+
B\left(\frac{D}{10^6}\right)^{-\beta}.
\]

Under the approximate training-compute constraint \(C \propto ND\), a stable additive fit would imply

\[
N_\star(C)\propto C^{\beta/(\alpha+\beta)},
\qquad
D_\star(C)\propto C^{\alpha/(\alpha+\beta)}.
\]

The script intentionally checks whether the pilot can identify this surface instead of forcing an attractive answer. A collapsed loss floor, boundary-seeking exponents, or poor log-space residuals mark the pilot fit as unstable. In that case, the correct conclusion is that denser multi-seed measurements are required.

## 5. Plot the measured pilot

In [ ]:
subprocess.run(
    [sys.executable, "experiments/dense_scaling/plot_results.py"],
    check=True,
)

from IPython.display import SVG, display

for figure_path in (
    "assets/dense_scaling_pilot_loss.svg",
    "assets/dense_scaling_pilot_exact_match.svg",
    "assets/dense_scaling_pilot_compute.svg",
):
    print(figure_path)
    display(SVG(filename=figure_path))

## 6. Inspect every measured point

In [ ]:
for record in sorted(
    complete,
    key=lambda item: (item["parameter_count"], item["steps"]),
):
    best = record["best"]
    print(
        f"{record['run_id']:<24} "
        f"N={record['parameter_count']:>10,} "
        f"D={record['sequence_tokens_seen']:>10,} "
        f"loss={best['loss']:.8f} "
        f"EM={best['greedy_exact_match']:.5f}"
    )

## 7. Re-run the pilot only when needed

Training is disabled by default because the repository already contains verified T4 artifacts. Enable it only to reproduce the experiment on a GPU runtime. The runner skips completed runs and supports model and step filters.

In [ ]:
RUN_TRAINING = False

if RUN_TRAINING:
    import jax

    print("Backend:", jax.default_backend())
    print("Devices:", jax.devices())
    assert jax.default_backend() == "gpu"

    subprocess.run(
        [
            sys.executable,
            "experiments/dense_scaling/train_grid.py",
            "--continue-on-error",
        ],
        check=True,
    )
else:
    print("Training skipped. Set RUN_TRAINING = True to reproduce the grid.")

## 8. Interpretation

The pilot supports four concrete conclusions:

1. **Fifty steps undertrain every model.** Their validation losses cluster around 1.7–1.9 and exact match is near zero.
2. **Capacity reduces the examples required to cross the algorithmic transition.** At 125 steps, the two largest models are already near perfect while the 162K model remains near zero exact match.
3. **Exact match is a phase-transition metric.** It changes abruptly and then becomes flat, so validation loss is the more useful response variable for scaling analysis.
4. **The finite task saturates.** Once the complete addition algorithm is learned, additional parameters and data mostly reduce confidence loss rather than increase task accuracy.

The additive Chinchilla surface is not forced when the pilot cannot identify it. The final experiment must add intermediate model sizes, multiple random seeds, and denser budgets around each model's transition. It should report both sequence tokens and supervised answer tokens, and it should keep the learning-rate protocol explicit.